<center><h1>Técnicas de Normalización de Datos</center>

<b>Maestría</b>: Inteligencia Artificial Aplicada <br>
<b>Asignatura:</b> Procesamiento Acelerado de Lenguaje Natural <br>
<b>Profesor:</b> Edwin J. Rueda

## Normalización

La normalización de datos consiste en transformar texto bruto en una representación más adecuada para su análisis. En este *Notebook* abordaremos las siguientes técnicas de normalización o procesamientos de texto:

- **Tokenización**
- **Lematización**
- **Stemming**
- **Lowercasing**
- **Eliminación de Stopwords**

### Tokenización
Tokenizar consiste el dividir texto en palabras o caracteres según sea la tarea que abarquemos. La tokenización se emplea como paso inicial al pre-procesar un texto y nos permite porteriormente:
- Analizar la estructura/morfología de cada palabra.
- Construir vocabularios para psoteriormente aplicar modelos estadisticos/redes neuronales/etc.

$$ tokenizar(\text{El dia está soleado}) = [\text{El, día, está, soleado}]$$

#### ¿ Por qué aplicar la tokenización ?

Tokenizar el corpus/texto, es la fase inicial de todo procesamiento de lenguaje natural. Ya que nuestros modelos próximos en donde utilizaremos nuestro corpus no entienden texto bruto, por lo cual debemos pre-procesar, normalizar/mapear nuestros datos y luego si darlos como entrada a determinado modelo estadístico, red neuronal, etc.

<center><img src="./images/Esquemas-Normalization.png"  height="200"></center>

#### Tokenización con Nalural Language Toolkit (NLTK)

In [2]:
import nltk
nltk.download('punkt_tab') #se descarga el mapeo del tokenizador en español

sentence = "el día en que todo comenzó, fue genial"
tokens = nltk.word_tokenize(sentence, language='spanish')
print(f"Podemos tokenizar textos: {sentence}")
print(f"tokens: {tokens} \n")

Podemos tokenizar textos: el día en que todo comenzó, fue genial
tokens: ['el', 'día', 'en', 'que', 'todo', 'comenzó', ',', 'fue', 'genial'] 



[nltk_data] Downloading package punkt_tab to /home/edwin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


**Nota:** Cómo podemos ver, después de toquenizado un corpus, podemos crear un diccionario de palabras y asignar a cada palabra un escalar, un vector, una matriz, etc. Lo que nos permitirá que nuestros modelos a entrenar, puedan recibir una entrada esperada.

$$map(token) = value$$

### Lematización
La lematización busca reducir una palabra a su forma canónica o raíz, teniendo en cuenta el contexto gramatical (su etiqueta morfosintáctica). A esto, se le conoce como **lema**.

$$Lema(\text{Que tan complejo es lematizar un texto en español}) \rightarrow \text{Que tanto complejo ser lematizar uno texto en español}$$ 

Para este tipo de procesamiento en corpus en español, *nltk* no tiene modelos disponibles. Por lo cual haremos uso de *Spacy*.

##### Proceso de lematización de una palabra
1) Se tokeniza el corpus/texto
2) Se etiqueta morfosintácticamente cada token/palabra del corpus.
3) Se genera el lema correcto en base a una base de datos léxica del modelo lingüistico.

#### ¿ Por qué lematizar ? 

Su principal funcionalidad es reducir la dimensionalidad de nuestro vocabulario para el posterior entrenamiento de modelos.

$$lema([\text{caminar, camina, caminó, caminaría, caminamos}]) \rightarrow \text{caminar}$$

**Nota**: En este ejemplo, reducimos la dimensionalidad de 5 a 1.

#### Lematización con *Spacy*
Utilizaremos un [pipeline](https://spacy.io/models/es) disponibilizado por *spacy*. Este pipeline nos permite realizar los 3 pasos mencionados anteriormente y nos genera un [documento](https://spacy.io/api/doc) con los respectivos **lemas**.

$$pipeline(corpus) \rightarrow Doc$$

##### ¿Qué es un documento en *spacy*?
- Un *Doc* es una clase que contiene una secuencia de **Tokens**. Donde a su vez, cada [token](https://spacy.io/api/token) (palabra, espacio en blanco, puntuación, etc) contiene una estructura individual (atributos) que podemos extraer de dicho token, como su **lema**.

In [2]:
# Descargamos el pipeline de spacy
!python3 -m spacy download es_core_news_sm

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 9.1 MB/s eta 0:00:00m eta 0:00:0100:011
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


In [3]:
import spacy
#cargamos el pipeline a utilizar
nlp_pipe = spacy.load("es_core_news_sm") 

- Ahora podemos lematizar un corpus/texto

In [4]:
doc = nlp_pipe("Que tan complejo es lematizar un texto en español")
print(f"Tipo de objeto generado: {type(doc)}")
for token in doc:
    print(f"{token.text:9} -> {token.lemma_}")

Tipo de objeto generado: <class 'spacy.tokens.doc.Doc'>
Que       -> que
tan       -> tanto
complejo  -> complejo
es        -> ser
lematizar -> lematizar
un        -> uno
texto     -> texto
en        -> en
español   -> español


### Stemming

Es el proceso por el cual se lleva un token/palabra a su raíz (*stem*). Realmente, su raíz puede que no sea una palabra propiamente, pero podríamos decir que es un "prefijo" de dicha palabra. Es un método muy rápido pero agresivo para normalizar texto.

$$stem(\text{saltar}) \rightarrow \text{salt}$$
$$stem(\text{camina}) \rightarrow \text{camin}$$
$$stem(\text{Que tan complejo es lematizar un texto en español}) \rightarrow ?$$

#### ¿Cómo se saca la raíz de una palabra/token? 
- Se aplican un conjunto de reglas para eliminar los sufijos.
- Este conjunto de reglas no dependen de la categoría gramatical. Son aplicadas a todos los *tokens* por igual.

#### ¿Por qué sacar el *stem*/raíz de una palabra/*token*?
- Es una forma rápida y sencilla de reducir la dimensionalidad de palabras dentro de un corpus y nos permite realizar busquedas rápidas dentro de documentos. No obstante, solo nos sirve en tareas donde realmente no necesitemos una precisión gramatical de las palabras.

#### Implementando Stem con *nltk*

El agoritmo empleado en *nltk* para sacar la raíz de las palabras es el ***Snowball***, nombrado así por su creador. Este [algoritmo](https://snowballstem.org/), consta de una [base de reglas](https://snowballstem.org/algorithms/spanish/stemmer.html) aplicadas para eliminar los sufijos de las palabras, dejando unicamente su raíz. No obstante, no preserva la categoria de la plabra, ni su significado original. Siendo que dos palabras podrían tener la misma raíz.

In [5]:
from nltk.stem.snowball import SnowballStemmer

# definimos el stemmer
stemmer = SnowballStemmer('spanish')
word = "complejo"
print(f"Palabra -> {word} | stem -> {stemmer.stem(word)}")

Palabra -> complejo | stem -> complej


- Podemos definir una función para sacar la raíz de las palabras dentro de una sentencia.

In [12]:
def get_stem_sentence(sentence):
    result = []
    for word in sentence.split():
        result.append(stemmer.stem(word))
    return result

sentence = "Que tan complejo es lematizar un texto en español"
print(f"Sentencia: {sentence}")
print(f"Sentencia raíz: {get_stem_sentence(sentence)}")

Sentencia: Que tan complejo es lematizar un texto en español
Sentencia raíz: ['que', 'tan', 'complej', 'es', 'lematiz', 'un', 'text', 'en', 'español']


**Nota**: Como podemos observar, sacar la raíz de una palabra es un método que es rápido, ya que implica aplicar una serie de reglas. Pero no mantiene el significado de dicha palabra. En ocasiones, puede provocar que dos palabras terminen teniendo la misma raíz.

### *Lowercasing* y *Stopwords*

*Lowercasing* y *stopwords* son dos conceptos fundamentales en NLP. El primero, es muy esencial y consiste en llevar todas las palabras de un corpus a minúsculas, esto debido a que si generamos tokens con un corpus que contenga mayúsculas, podríamos generar dos o más tokens para una misma palabra.

<center>Ella, ella, ELLA -> son 3 tokens diferentes</center>

Aunque suene lógico siempre llevar los corpus a palabras en minúsculas, para tareas de reconocimiento de identidades, no se debería aplicar *lowercasing*, debido a que los nombres propios comienzan con mayúscula. Entonces sería muy útil conservar estas letras mayúsculas.

Por otro lado, las ***stopwords*** consisten en palabras que aparecen con mucha frecuencia en un determinado idioma y no suelen aportar valor (articulos, preposiciones, adverbios, etc).

#### ¿Por qué eliminar las stopwords?

- Como mencionabamos anteriormente, al igual que en la lematización o stemmer. Eliminar las **stopwords** nos permite reducir la dimensionalidad de nuestro corpus. Lo que nos permitiría mas adelante disminuir el tiempo de entrenamiento de modelos de AI.
- Reducimos ruido de nuestro conjunto de datos (Nota: solo si las  *stopwords* no son necesarias en nuestra tarea).

**Nota:** Tener en cuenta que no siempre es útil eliminar *stopwords*. Por ejemplo, en tareas de análisis de sentimiento, resulta útil mantener el adverbio **No** para saber si a una persona no le gusta determinada cosa.


#### Implementemos *lowercasing* y *stopwords*
- La implementación de lowercasing en python resulta realmente sencillo. Esto debido a que los *string* en python cuentan con la propiedad ***lower***, la cual retorna en minúsculas el *string*.

In [13]:
sentence = "Esta ORACIÓN será LLEVADA a MINÚSCULAS"
print(f"Sentencia inicial: {sentence}")
sentence = sentence.lower()
print(f"Sentencia procesada: {sentence}")

Sentencia inicial: Esta ORACIÓN será LLEVADA a MINÚSCULAS
Sentencia procesada: esta oración será llevada a minúsculas


- Para el filtrado de *stopwords* nos podemos apoyar de *nltk*

In [14]:
from nltk.corpus import stopwords
import numpy as np

stop_words = stopwords.words("spanish")
print(f"Cinco stopwords: {np.random.choice(stop_words, size=5)}")

Cinco stopwords: ['cual' 'estén' 'hube' 'o' 'sentid']


- Cuando tokenizamos, podemos filtrar nuestras palabras para eliminar las *stopwords*

In [15]:
tokens = nltk.word_tokenize(sentence, language='spanish', )
new_tokens = [word for word in tokens if word not in stop_words]
print(f"Sentencia inicial: {sentence}")
print(f"Sentencia procesada: {new_tokens}")

Sentencia inicial: esta oración será llevada a minúsculas
Sentencia procesada: ['oración', 'llevada', 'minúsculas']


### Signos de puntuación o especiales

En algunos casos los signos de puntuación agregan ruido a nuestro corpus, aumentando su complejidad. Aunque es clave en casos como el análisis de sentimientos.

##### ¿Cómo podemos eliminar signos de puntución o caracteres especiales?

In [51]:
import string

sentence = "tenemos % c4racteres & / espci#ciales!!"
print(string.punctuation)
sentence.translate(str.maketrans('','',string.punctuation))

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


'tenemos  c4racteres   espciciales'

In [52]:
def rm_special_characters(sentence):
    return sentence.translate(str.maketrans('','',string.punctuation))

In [53]:
import re

re.sub(r'[^a-zA-Z0-9\s]','',sentence)

'tenemos  c4racteres   espciciales'

### Apliquemos lo visto al corpus CESS-ESP
Para este ejemplo, solo cargaremos las sentencias del corpus, sin su etiqueta morfológica. Lo que haremos es dos filtrados.
1) Filtrado basado en lematización:
    - Para el filtrado, haremos los siguiente:
      1) Aplicaremos lowercassing a todas las palabras
      2) Eliminaremos las *stopwords*
      3) Lematizamos todos los tokens
<br><br>
2) Filtrado basado en stemmer:
   - procesaremos así:
     1) lowercassing a todas las palabras
     2) Eliminación de *stopwrods*
     3) stemmer de los tokens
    

Al final, comparamos reducción de dimensionalidad y tiempos de ejecución del procesamiento

- Definimos una función para cargar los datos

In [10]:
from nltk.corpus import cess_esp

def load_words():
    return cess_esp.words()

words = load_words()

In [11]:
print(f"Primeros 10 tokens:\n {words[:10]}")

Primeros 10 tokens:
 ['El', 'grupo', 'estatal', 'Electricité_de_France', '-Fpa-', 'EDF', '-Fpt-', 'anunció', 'hoy', ',']


##### Filtrado basado en lematización

In [12]:
nlp_pipe = spacy.load("es_core_news_sm") 
stop_words = stopwords.words("spanish")

def apply_lemma_filter(tokens):
    result = set()
    for token in tokens:
        lw_token = token.lower() # lowercase mode
        if lw_token not in stop_words:
            lemma = nlp_pipe(lw_token).doc[0].lemma_
            result.add(lemma)
    return result

In [13]:
from time import time

tic = time()
print("procesando y normalizando las palabras...")
unique_words = len(words)
print(f"Cantidad de palabras únicas antes del procesamiento: {unique_words}")
new_lema_words = apply_lemma_filter(words)
unique_lema_words = len(new_lema_words)
print(f"Cantidad de palabras únicas después del procesamiento: {unique_lema_words}")
print(f"Reducción de la dimensionalidad: {(1 - (unique_lema_words/unique_words))}")
toc = time()
print(f"Tiempo de procesamiento: {(toc-tic)/60} [min]")

procesando y normalizando las palabras...
Cantidad de palabras únicas antes del procesamiento: 192686
Cantidad de palabras únicas después del procesamiento: 17690
Reducción de la dimensionalidad: 0.9081926035103745
Tiempo de procesamiento: 9.089685030778249 [min]


##### Filtro basado en stemmer

In [14]:
stemmer = SnowballStemmer('spanish')
stop_words = stopwords.words("spanish")

def apply_stemmer_filter(tokens):
    result = set()
    for token in tokens:
        lw_token = token.lower() # lowercase mode
        if lw_token not in stop_words:
            stem = stemmer.stem(lw_token)
            result.add(stem)
    return result

In [15]:
tic = time()
print("procesando y normalizando las palabras...")
unique_words = len(words)
print(f"Cantidad de palabras únicas antes del procesamiento: {unique_words}")
new_stem_words = apply_stemmer_filter(words)
unique_stem_words = len(new_lema_words)
print(f"Cantidad de palabras únicas después del procesamiento: {unique_stem_words}")
print(f"Reducción de la dimensionalidad: {(1 - (unique_stem_words/unique_words))}")
toc = time()
print(f"Tiempo de procesamiento: {(toc-tic)/60} [min]")

procesando y normalizando las palabras...
Cantidad de palabras únicas antes del procesamiento: 192686
Cantidad de palabras únicas después del procesamiento: 17690
Reducción de la dimensionalidad: 0.9081926035103745
Tiempo de procesamiento: 0.15214710235595702 [min]


### Conclusiones

- El aplicar técnicas de NLP nos permite reducir dimensionalidad de nuestros datos, lo que disminuye el tiempo de entrenamiento de modelos de AI.
- También nos permite eliminar palabras que no otorgan importancia en nuestro corpus.
- No existe un receta única para el procesado del corpus, depende de varios factores.
- Si de puede desistir del significado real de cada palabra, utilizar *stemming* reduce el tiempo de procesamiento del corpus.

#### Requisitos:
- Spacy 3.7.0

### Documentación
- https://www.nltk.org/index.html
- https://snowballstem.org/algorithms/spanish/stemmer.html